# 05 — Synchronised farm replay (presentation-only)

This notebook is separate from `04_algorithm_comparison.ipynb`. It reuses the project's audited simulator and seed-11 final checkpoints, with no training.

**Important:** 2017 is a *validation* season, not one of the held-out test seasons (2021–2024). The replay illustrates policy behaviour; use the comparison notebook's test-year results for claims about generalisation.

Run from repository root or the `notebooks/` directory; the outputs go to `outputs/replay/`.

In [ ]:
from pathlib import Path
import json, ast, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import torch

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebooks/00_shared_template.ipynb').exists()), None)
if ROOT is None: raise FileNotFoundError('Clone the project repository first.')
sys.path.insert(0, str(ROOT))
OUT = ROOT/'outputs/replay'
OUT.mkdir(parents=True, exist_ok=True)
YEAR, SEED, FPS = 2017, 11, 8
print('Project root:', ROOT)

In [ ]:
# Execute only the fixed shared simulator definitions; never run training or test loops.
def cell_source(name, i):
    obj=json.loads((ROOT/'notebooks'/name).read_text())
    return ''.join(obj['cells'][i].get('source', []))

shared={}
for i in (6,8,10,13,20):
    exec(compile(cell_source('00_shared_template.ipynb',i), f'shared_cell_{i}', 'exec'), shared)
# Extract the threshold benchmark function only, not the benchmark execution.
t=ast.parse(cell_source('00_shared_template.ipynb',22))
for node in t.body:
    if isinstance(node,ast.FunctionDef) and node.name=='threshold_policy':
        exec(compile(ast.Module(body=[node],type_ignores=[]),'threshold_policy','exec'),shared)
print('Simulator and baseline policy loaded')

In [ ]:
# Reuse trained model architecture definitions exactly as implemented in each algorithm notebook.
# Only declarative definitions are loaded: training cells, optimizer loops, and replay cells are not run.
def model_namespace(filename):
    doc=json.loads((ROOT/'notebooks'/filename).read_text())
    ns=dict(shared)
    included=[]
    for cell in doc['cells']:
        if cell.get('cell_type')!='code': continue
        src=''.join(cell.get('source',[]))
        try: module=ast.parse(src)
        except SyntaxError: continue
        for node in module.body:
            if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and (node.name.endswith(('Agent','Actor','Critic','Network')) or node.name in ('build_mlp','adam','soft_update')):
                included.append(node)
    for node in included:
        try: exec(compile(ast.Module(body=[node],type_ignores=[]),filename,'exec'),ns)
        except NameError as exc: raise RuntimeError(f'Unresolved model dependency {getattr(node,"name",node)} in {filename}: {exc}') from exc
    return ns

NAMESPACES={a:model_namespace(n) for a,n in [('DDPG','01_ddpg.ipynb'),('TD3','02_td3.ipynb'),('SAC','03_sac.ipynb')]}
print('Loaded model definitions:', {k:[x for x in v if x.endswith('Agent')][-4:] for k,v in NAMESPACES.items()})

In [ ]:
# Load each saved checkpoint without modifying it. These files are created by our own training notebooks.
CHECKPOINTS={
 'DDPG': ROOT/'outputs/ddpg/final/ddpg_seed11.pt',
 'TD3': ROOT/'outputs/td3/final/td3_seed11.pt',
 'SAC': ROOT/'outputs/sac/final/sac_final_seed11.pt',
}
for name,p in CHECKPOINTS.items():
    if not p.exists(): raise FileNotFoundError(p)
    ck=torch.load(p, map_location='cpu', weights_only=False)
    print(name, 'checkpoint keys:', list(ck)[:24] if isinstance(ck,dict) else type(ck))

In [ ]:
# Checkpoint restore adapters: require exact actor weights before running; never substitute random networks.
def load_policy(name):
    ns=NAMESPACES[name]
    cp=torch.load(CHECKPOINTS[name],map_location='cpu',weights_only=False)
    actor_keys=('actor','actor_state_dict','actor_state','policy_state_dict')
    state=next((cp[k] for k in actor_keys if isinstance(cp,dict) and k in cp and isinstance(cp[k],dict)),None)
    if state is None:
        raise ValueError(f'{name}: unsupported checkpoint structure {list(cp) if isinstance(cp,dict) else type(cp)}; inspect keys above before updating adapter')
    # Select architecture from the original notebook, never recreate by guesswork.
    actor_name={'DDPG':'DeterministicActor','TD3':'DeterministicActor','SAC':'SACActor'}[name]
    cls=ns.get(actor_name)
    if cls is None: raise ValueError(f'{name}: {actor_name} unavailable; inspect model definitions')
    model=cls(18,1).cpu()
    model.load_state_dict(state,strict=True)
    model.eval()
    def policy(obs):
        with torch.no_grad():
            features=torch.as_tensor(obs,dtype=torch.float32).reshape(1,-1)
            if name=='SAC':
                z=torch.tanh(model.mean(model.shared(features)))
            else:
                z=model(features)
                if isinstance(z,tuple): z=torch.tanh(z[0])
            return z.cpu().numpy().ravel().astype(np.float32)
    return policy

POLICIES={k:load_policy(k) for k in CHECKPOINTS}
POLICIES['Threshold']=shared['threshold_policy']
print('Loaded policies with strict checkpoint matching.')

In [ ]:
# Full 150-day synchronized traces: same weather, initial water, 2017 validation season.
def trace(name,policy):
    env=shared['KyabramIrrigationEnv'](split='validation')
    obs,_=env.reset(options={'season_year':YEAR})
    rows=[]; discounted=0.
    for day in range(150):
        action=np.asarray(policy(obs),dtype=np.float32).reshape(1)
        obs,reward,terminated,truncated,info=env.step(action)
        discounted += shared['SHARED_TRAINING']['gamma']**day * reward
        rows.append({'agent':name,'day':day+1,'date':info['decision_date'],'soil_mm':info['storage_after_mm'],
                     'minimum_mm':info['minimum_storage_mm'],'irrigation_mm':info['irrigation_mm'],
                     'rain_mm':info['rain_mm'],'drainage_mm':info['drainage_mm'],'reward':reward})
    assert terminated and not truncated
    df=pd.DataFrame(rows)
    df['cumulative_irrigation_mm']=df['irrigation_mm'].cumsum()
    return df,{'agent':name,'year':YEAR,'seed':SEED if name!='Threshold' else None,
               'discounted_return':discounted,'irrigation_mm':df['irrigation_mm'].sum(),
               'days_below_target':int((df['soil_mm']<df['minimum_mm']).sum())}

TRACES=[]; METRICS=[]
for name,policy in POLICIES.items():
    df,metrics=trace(name,policy);TRACES.append(df);METRICS.append(metrics)
replay=pd.concat(TRACES,ignore_index=True)
replay.to_csv(OUT/'farm_replay_2017_daily.csv',index=False)
pd.DataFrame(METRICS).to_csv(OUT/'farm_replay_2017_summary.csv',index=False)
display(pd.DataFrame(METRICS))

In [ ]:
# 37.5 second PowerPoint-ready animation, synchronized across 4 panels.
COLORS={'DDPG':'#3C7BB6','TD3':'#E58A40','SAC':'#6E62B5','Threshold':'#73837C'}
fig, axs=plt.subplots(2,2,figsize=(12.8,7.2),sharex=True,layout='constrained')
fig.patch.set_facecolor('white')
line_cache={}
for ax,(name,df) in zip(axs.flat,zip(POLICIES,replay.groupby('agent',sort=False))):
    ax.set_title(name,fontweight='bold',color=COLORS[name]); ax.set_xlim(1,150);ax.set_ylim(0,115)
    ax.set_xlabel('Day of maize season');ax.set_ylabel('Soil water (mm)')
    ax.grid(alpha=.15)
    soil,=ax.plot([],[],color=COLORS[name],lw=2.6,label='Soil water')
    lower,=ax.plot([],[],color='#CF5959',ls='--',lw=1.3,label='Minimum target')
    box=ax.text(.02,.02,'',transform=ax.transAxes,va='bottom',fontsize=9)
    if name=='DDPG':ax.legend(loc='upper right',fontsize=8)
    line_cache[name]=(soil,lower,box,replay[replay.agent==name].reset_index(drop=True))
fig.suptitle('ONE SEASON · SAME WEATHER · FOUR IRRIGATION STRATEGIES',fontsize=15,fontweight='bold')
def frame(i):
    for name,(soil,lower,box,df) in line_cache.items():
        sub=df.iloc[:i+1]
        soil.set_data(sub.day,sub.soil_mm);lower.set_data(sub.day,sub.minimum_mm)
        row=sub.iloc[-1]
        box.set_text(f'Day {int(row.day):03}  |  Today: {row.irrigation_mm:.1f} mm  |  Cumulative: {row.cumulative_irrigation_mm:.0f} mm\\nDrainage today: {row.drainage_mm:.1f} mm')
    return []
anim=FuncAnimation(fig,frame,frames=150,interval=250,blit=False)
try:
    anim.save(str(OUT/'farm_replay_2017.mp4'),writer=FFMpegWriter(fps=4,bitrate=2500))
    print('Exported 37.5-second MP4')
except Exception as exc:
    print('MP4 encoding unavailable:',exc)
frame(149)
fig.savefig(OUT/'farm_replay_2017_preview.png',dpi=140,bbox_inches='tight')
plt.show()